# B"H
# Final Project in Medical Imaging
## GUI Layout Presentation
---
### Supervisor: Dr. Eyal Ben-Issac
### Students: Lior Cohen & Tomer Peretz
### Date: 20.07.2025

---
**You may only use the Interface part**
---
**It takes about 4 minutes to load all the models**

## Loading

### Imports

> Import libraries

In [ ]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

# Authenticate and create the PyDrive client
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

In [ ]:
!pip install torchxrayvision
from IPython.display import clear_output
clear_output()

In [ ]:
import re
import cv2
import pickle
import torch
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import requests
import torchvision
from PIL import Image
from io import BytesIO
import tensorflow as tf
import ipywidgets as widgets
import torchxrayvision as xrv
import matplotlib.pyplot as plt
from tensorflow.keras import models, layers
from IPython.display import display, HTML, Audio

### Models

> Load the models from Google Drive

*   Define a mapping of model filenames to their loader functions and types (CNN, ML, TORCH)
*   Support loading for Keras (.keras), joblib (.pkl), and PyTorch (.pt) models

In [ ]:
model_folder_id = '1x4b4X-meKrRaD5CrXzjUsabrH0OxKvql'
file_list = drive.ListFile({'q': f"'{model_folder_id}' in parents and trashed=false"}).GetList()

In [ ]:
def load_cnn_model(filename):
    for file in file_list:
        if file['title'] == filename:
            file.GetContentFile(filename)
            print(f"Downloaded {filename} successfully!")
            return tf.keras.models.load_model(filename)


def load_joblib_model(filename):
    for file in file_list:
        if file['title'] == filename:
            file.GetContentFile(filename)
            print(f"Downloaded {filename} successfully!")
            with open(filename, 'rb') as f:
                model = joblib.load(f)
            return model


def load_torch_model(filename):
    if filename in ["CheXpert.pt", "TorchXRayVision.pt"]:
        torch_folder_id = model_folder_id
        torch_list = file_list
    else:
        torch_folder_id = '1cKdMIE3Jmc-ZwrEghZQNhBqx6GWeleyf'
        torch_list = drive.ListFile({'q': f"'{torch_folder_id}' in parents and trashed=false"}).GetList()
    for file in torch_list:
        if file['title'] == filename:
            file.GetContentFile(filename)
            print(f"Downloaded {filename} successfully!")
            model = torch.load(filename, weights_only=False)
            model.eval()
            return model

In [ ]:
model_loaders = {
    "lor.pkl": (load_joblib_model, "ML"),
    "ensemble.pkl": (load_joblib_model, "ML"),
    "xgb.pkl": (load_joblib_model, "ML"),
    "svm.pkl": (load_joblib_model, "ML"),
    "rf.pkl": (load_joblib_model, "ML"),
    "cnn.keras": (load_cnn_model, "CNN"),
    "vgg19.keras": (load_cnn_model, "CNN"),
    "TorchXRayVision.pt": (load_torch_model, "TORCH"),
    "CheXpert.pt": (load_torch_model, "TORCH"),
    "densenet121-res224-all.pt": (load_torch_model, "TORCH"),
    "densenet121-res224-nih.pt": (load_torch_model, "TORCH"),
    "densenet121-res224-chex.pt": (load_torch_model, "TORCH"),
    "densenet121-res224-rsna.pt": (load_torch_model, "TORCH"),
    "densenet121-res224-mimic_nb.pt": (load_torch_model, "TORCH"),
    "densenet121-res224-mimic_ch.pt": (load_torch_model, "TORCH"),
    "chexpert-custom.pt": (load_torch_model, "TORCH"),
}



*   Main loop to load all the models



In [ ]:
loaded_models = {}

for filename, (loader_func, model_type) in model_loaders.items():
    try:
        model = loader_func(filename)
        name = filename.split('.')[0].split('_')[0]
        loaded_models[name] = {"model": model, "type": model_type}
    except Exception as e:
        print(f"❌ Failed to load {filename}: {e}")

Downloaded lor.pkl successfully!
Downloaded ensemble.pkl successfully!
Downloaded xgb.pkl successfully!
Downloaded svm.pkl successfully!
Downloaded rf.pkl successfully!
Downloaded cnn.keras successfully!
Downloaded vgg19.keras successfully!
Downloaded TorchXRayVision.pt successfully!
Downloaded CheXpert.pt successfully!
Downloaded densenet121-res224-all.pt successfully!
Downloaded densenet121-res224-nih.pt successfully!
Downloaded densenet121-res224-chex.pt successfully!
Downloaded densenet121-res224-rsna.pt successfully!
Downloaded densenet121-res224-mimic_nb.pt successfully!
Downloaded densenet121-res224-mimic_ch.pt successfully!
Downloaded chexpert-custom.pt successfully!


In [ ]:
loaded_models

{'lor': {'model': LogisticRegression(max_iter=1000, random_state=42),
  'type': 'ML'},
 'ensemble': {'model': VotingClassifier(estimators=[('rf', RandomForestClassifier(random_state=42)),
                               ('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=None, device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric='mlogloss',
                                              feature_types=None, gamma=None,
                                              grow_policy=None,
                                              importan.

*   Store loaded models in a structured dictionary categorized by type

In [ ]:
torch_models = {k: v for k, v in loaded_models.items() if v["type"] == "TORCH"}
cnn_models = {k: v for k, v in loaded_models.items() if v["type"] == "CNN"}
ml_models = {k: v for k, v in loaded_models.items() if v["type"] == "ML"}

In [ ]:
transform = torchvision.transforms.Compose([
    xrv.datasets.XRayCenterCrop(),
    xrv.datasets.XRayResizer(224)
])

## Utils

### Images

> Utility functions for preparing PIL images for different model types

  *   Convert PIL images to grayscale or RGB
  *   Resize and clean image using thresholding and inpainting
  *   Normalize and format into tensors or numpy arrays
  *   Ensure compatibility with CNN, VGG, and TorchXRayVision models

In [ ]:
def preprocess_pil_image(pil_img):
    img = pil_img.convert('RGB')
    img = img.resize((224, 224))
    img_np = np.array(img).astype('float32')
    if img_np.ndim == 3:
        img_np = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    _, mask = cv2.threshold(img_np, 230, 255, cv2.THRESH_BINARY)
    mask = mask.astype(np.uint8)
    img_np = cv2.inpaint(img_np.astype(np.uint8), mask, inpaintRadius=3, flags=cv2.INPAINT_TELEA)
    img_np = img_np.astype('float32')
    if img_np.max() > 1.1:
        img_np = img_np / 255.0
    img_np = xrv.datasets.normalize(img_np, 1.0)
    img_np = img_np[None, ...]
    img_np = transform(img_np)
    tensor_img = torch.from_numpy(img_np).unsqueeze(0)
    return tensor_img


def process_pil_image_list(pil_images, model):
    results = []
    for i, pil_img in enumerate(pil_images):
        try:
            input_tensor = preprocess_pil_image(pil_img)
            with torch.no_grad():
                output = model(input_tensor)
            preds = torch.sigmoid(output).numpy().flatten()
            pred_dict = dict(zip(model.pathologies, preds))
            results.append(pred_dict)
        except Exception as e:
            print(f"⚠️ Image {i} failed: {e}")
    return results


def preprocess_pil_image_for_xrv(pil_img):
    if pil_img.mode == 'RGBA':
        pil_img = pil_img.convert('RGB')
    pil_img = pil_img.convert('L')
    np_img = np.array(pil_img)
    np_img = xrv.datasets.normalize(np_img, 255)
    np_img = np_img[None, ...]
    np_img = transform(np_img)
    tensor_img = torch.from_numpy(np_img).unsqueeze(0)
    return tensor_img

In [ ]:
def process_image(image):
    img_np = np.array(image).astype('float32') / 255.0
    img_uint8 = (img_np * 255).astype(np.uint8)
    if img_uint8.ndim == 3:
        img_uint8 = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)
    _, mask = cv2.threshold(img_uint8, 230, 255, cv2.THRESH_BINARY)
    mask = mask.astype(np.uint8)
    cleaned = cv2.inpaint(img_uint8, mask, inpaintRadius=3, flags=cv2.INPAINT_TELEA)
    cleaned = cleaned.astype('float32') / 255.0
    cleaned_resized = cv2.resize(cleaned, (128, 128))
    input_tensor = np.expand_dims(cleaned_resized, axis=(0, -1))
    return input_tensor

def process_image_vgg(image):
    IMG_SIZE = (224, 224)
    if image.mode != 'RGB':
        image = image.convert('RGB')
    image = image.resize(IMG_SIZE)
    image_array = np.array(image).astype('float32') / 255.0
    image_array = np.expand_dims(image_array, axis=0)
    return image_array

### Models

> Run the models on a given image and return predictions

*   Keras models (CNN and VGG) use processed image tensors and return predicted folder and probability
*   ML models use extracted features and return predicted folder and confidence
*   Torch models (X-ray pretrained) output disease labels with probabilities
*   Aggregation functions summarize results across multiple models to produce a final decision

In [ ]:
def run_keras_models(image):
    results = []
    keras_folder_labels = {0: '01', 1: '02', 2: '03'}
    for name, obj in cnn_models.items():
        model = obj["model"]
        if name == "vgg19":
            input_tensor = process_image_vgg(image)
        else:
            input_tensor = process_image(image)
        pred = model.predict(input_tensor)
        class_idx = int(np.argmax(pred))
        results.append({
            "Model": name.upper(),
            "Predicted Folder": keras_folder_labels.get(class_idx, f"Class {class_idx}"),
            "Probability": float(pred[0][class_idx])
        })
    return results

In [ ]:
feature_extractor = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 1)),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
    ])


def aggregate_ml_predictions(ml_results):
    folder_scores = {}
    folder_counts = {}
    for result in ml_results:
        folder = result["Predicted Folder"]
        prob = float(result["Probability"])
        if folder not in folder_scores:
            folder_scores[folder] = 0.0
            folder_counts[folder] = 0
        folder_scores[folder] += prob
        folder_counts[folder] += 1
    avg_scores = {folder: folder_scores[folder] / folder_counts[folder] for folder in folder_scores}
    final_folder = max(avg_scores, key=avg_scores.get)
    final_score = avg_scores[final_folder]
    total_score = sum(avg_scores.values())
    confidence = final_score / total_score if total_score > 0 else 0
    return final_folder, final_score, confidence


def run_ml_models(image):
    results = []
    keras_folder_labels = {'0': '01', '1': '02', '2': '03'}
    input_tensor = process_image(image)
    features = feature_extractor(input_tensor)[0].numpy()
    features_2d = features.reshape(1, -1)
    for name, obj in ml_models.items():
        model = obj["model"]
        pred = model.predict(features_2d)[0]
        proba = model.predict_proba(features_2d)[0][pred] if hasattr(model, "predict_proba") else None
        folder_label = keras_folder_labels.get(str(pred))
        results.append({
            "Model": name.upper(),
            "Predicted Folder": folder_label,
            "Probability": float(proba) if proba is not None else "N/A",
        })
    return results

In [ ]:
def aggregate_disease_predictions(torch_results):
    disease_scores = {}
    disease_counts = {}
    for result in torch_results:
        disease = result.get("Predicted Disease", "").strip()
        if disease == "":
            disease = "No Finding"
        prob = float(result.get("Probability", 0))
        if disease not in disease_scores:
            disease_scores[disease] = 0.0
            disease_counts[disease] = 0
        disease_scores[disease] += prob
        disease_counts[disease] += 1
    avg_scores = {
        disease: disease_scores[disease] / disease_counts[disease]
        for disease in disease_scores
    }
    disease_scores_weighted = {
        disease: (avg_scores[disease] * disease_counts[disease])
        for disease in avg_scores
    }
    final_disease = max(disease_scores_weighted, key=disease_scores_weighted.get)
    final_score = avg_scores[final_disease]
    return final_disease, final_score, avg_scores


def run_pytorch_models(image):
    results = []
    tensor_img = preprocess_pil_image(image)
    for name, obj in torch_models.items():
        model = obj["model"]
        with torch.no_grad():
            output_tensor = model(tensor_img)
            probs = torch.sigmoid(output_tensor).numpy().flatten()
            pred_class = np.argmax(probs)
            predicted = model.pathologies[pred_class] if hasattr(model, 'pathologies') else f"Class {pred_class}"
            predicted = predicted.strip()
            if predicted == "":
                predicted = "No Finding"
            results.append({
                "Model": name,
                "Predicted Disease": predicted,
                "Probability": float(probs[pred_class])
            })
    return results

### Plots & Display

> Combine predictions and visualize final results

*   Compute weighted decision from multiple models to choose final folder
*   Display final classification result with styled HTML output
*   Plot bar graphs of folder prediction scores based on model weights
*   Plot bar graphs of disease scores from PyTorch models

In [ ]:
model_weights = {"CNN": 1.7, "ML": 1.1}

def decide_final_folder(df):
    df_copy = df.copy()
    def get_weight(name):
        name = name.lower()
        if "aggregated" in name:
            return model_weights["ML"]
        elif name.startswith("cnn") or name.startswith("vgg"):
            return model_weights["CNN"]
        else:
            return 1
    df_copy["Weight"] = df_copy["Model"].apply(get_weight)
    df_copy["Probability"] = pd.to_numeric(df_copy["Probability"], errors="coerce").fillna(0)
    df_copy["Score"] = df_copy["Probability"] * df_copy["Weight"]
    folder_scores = df_copy.groupby("Predicted Folder")["Score"].sum()
    final_prediction = folder_scores.idxmax()
    final_score = folder_scores.max()
    total_score = folder_scores.sum()
    confidence = final_score / total_score if total_score > 0 else 0
    return final_prediction, final_score, confidence

In [ ]:
def display_final_prediction(folder, score, confidence):
    color_map = {
        '01': '#1565c0',
        '02': '#880e4f',
        '03': '#2e7d32'
    }
    color = color_map.get(folder, "#424242")
    percent = f"{confidence * 100:.1f}%"
    html = f"""
    <div style="
        background-color: #1e1e1e;
        color: #f0f0f0;
        border-left: 8px solid {color};
        padding: 18px;
        font-size: 18px;
        font-family: 'Segoe UI', sans-serif;
        margin-bottom: 20px;
        border-radius: 6px;
        max-width: 290px;
        box-shadow: 0 2px 6px rgba(0,0,0,0.5);
    ">
        <div style="font-size: 22px; font-weight: bold;">
            Final Predicted Folder: <span style="color: {color};">{folder}</span>
        </div>
        <div>
            Weighted Confidence: <strong>{percent}</strong>
        </div>
    </div>
    """
    display(HTML(html))

In [ ]:
def plot_folder_scores(df):
    folder_colors = {'01': '#1565c0', '02': '#880e4f', '03': '#2e7d32'}
    df_copy = df.copy()
    df_copy["Weight"] = df_copy["Model"].apply(lambda name: model_weights["CNN"] if name.startswith("cnn") else model_weights["ML"])
    df_copy["Probability"] = pd.to_numeric(df_copy["Probability"], errors="coerce").fillna(0)
    df_copy["Score"] = df_copy["Probability"] * df_copy["Weight"]
    folder_scores = df_copy.groupby("Predicted Folder")["Score"].sum().sort_index()
    colors = [folder_colors.get(folder, '#999999') for folder in folder_scores.index]
    plt.figure(figsize=(4, 2.5))
    folder_scores.plot(kind='bar', color=colors, edgecolor='black')
    plt.title("Weighted Folder Scores", fontsize=11)
    plt.xlabel("Folder", fontsize=10)
    plt.ylabel("Score", fontsize=10)
    plt.xticks(rotation=0, fontsize=9)
    plt.yticks(fontsize=9)
    plt.grid(axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_disease_scores(pytorch_results):
    disease_scores = {}
    disease_counts = {}
    for result in pytorch_results:
        disease = result.get("Predicted Disease", "").strip()
        prob = float(result.get("Probability", 0))
        if disease not in disease_scores:
            disease_scores[disease] = 0.0
            disease_counts[disease] = 0
        disease_scores[disease] += prob
        disease_counts[disease] += 1
    weighted_scores = {
        disease: round(disease_scores[disease], 3)
        for disease in disease_scores
    }
    sorted_items = sorted(weighted_scores.items(), key=lambda x: x[1], reverse=True)
    diseases, scores = zip(*sorted_items)
    palette = sns.color_palette("hls", len(diseases))
    color_dict = dict(zip(diseases, palette))
    colors = [color_dict[d] for d in diseases]
    plt.figure(figsize=(6, 3))
    bars = plt.bar(diseases, scores, color=colors, edgecolor='black')
    plt.title("Weighted Disease Scores", fontsize=12)
    plt.ylabel("Score", fontsize=10)
    plt.xticks(rotation=45, ha='right', fontsize=9)
    for bar, score in zip(bars, scores):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f"{score:.3f}",
                 ha='center', va='bottom', fontsize=8)
    plt.tight_layout()
    plt.show()

### Execution

> Run the models and generate predictions

*   Run Keras and ML models on the uploaded image
*   Combine predictions, aggregate ML results, and decide final folder
*   Display folder classification result and show summary tables
*   Run PyTorch disease detection models and show final predicted disease
*   Plot disease bar chart and show detailed predictions



In [ ]:
def run_models_on_image(image: Image.Image):
    output.clear_output()
    with output:
        keras_results = run_keras_models(image)
        ml_results = run_ml_models(image)
        clear_output()
        print("\n\n")
        ml_folder, ml_score, ml_confidence = aggregate_ml_predictions(ml_results)
        ml_aggregated_result = {
            "Model": "CNN + ML (Aggregated)",
            "Predicted Folder": ml_folder,
            "Probability": ml_score
        }
        df1 = pd.DataFrame(keras_results + [ml_aggregated_result])
        final_folder, final_score, confidence = decide_final_folder(df1)
        display_final_prediction(final_folder, final_score, confidence)
        plot_folder_scores(df1)
        print("\n\nClassifiy Folder:\n")
        display(df1)
        print("\n\nFull ML results:\n")
        df3 = pd.DataFrame(ml_results)
        display(df3)

In [ ]:
def run_detector_on_image(image: Image.Image):
    output.clear_output()
    with output:
        pytorch_results = run_pytorch_models(image)
        clear_output()
        print("\n\n")
        final_disease, disease_score, _ = aggregate_disease_predictions(pytorch_results)
        df2 = pd.DataFrame([{
    "Model": "Disease-Aggregated",
    "Predicted Disease": final_disease,
    "Probability": round(disease_score, 3)
}])
        plot_disease_scores(pytorch_results)
        print("\n\nDisease Detector:\n")
        display(df2)
        print("\n\nFull Diseases results:\n")
        df4 = pd.DataFrame(pytorch_results)
        display(df4)

## Layout

### Widgets

> Set up the interface elements and interactions

*   Define buttons, text fields, and upload widget
*   Set behavior when URL is entered or image is uploaded
*   Handle button clicks to run classification, detection, and image display
*   Clear and reset widgets as needed

In [ ]:
upload_widget = widgets.FileUpload(accept='.jpg,.jpeg,.png,.JPG,.JPEG,.PNG', multiple=False, description="⬆️ Upload")
url_widget = widgets.Text(placeholder='🌐 Image URL', layout=widgets.Layout(width='150px'))
run_button = widgets.Button(description='▶️ Run')
reset_button = widgets.Button(description='🔄 Reset')
detector_button = widgets.Button(description='🦠 Detector')
gradcam_button = widgets.Button(description='🌌 Explain')
show_image_button = widgets.Button(description='🧿 View')
output = widgets.Output()

In [ ]:
def on_reset_clicked(b):
    upload_widget._counter = 0
    upload_widget.value.clear()
    upload_widget.disabled = False
    url_widget.value = ""
    url_widget.disabled = False
    output.clear_output()

In [ ]:
def convert_drive_url(original_url):
    match = re.search(r"/d/([\w-]+)", original_url)
    if match:
        file_id = match.group(1)
        return f"https://drive.google.com/uc?export=download&id={file_id}"
    return original_url

In [ ]:
def on_url_change(change):
    if change['new']:
        upload_widget.disabled = True
    else:
        upload_widget.disabled = False

In [ ]:
def on_upload_change(change):
    if upload_widget.value:
        upload_widget.disabled = True
        url_widget.disabled = True
    else:
        upload_widget.disabled = False
        url_widget.disabled = False

In [ ]:
def on_click_run(b):
    image = None
    if upload_widget.value:
        uploaded_file = next(iter(upload_widget.value.values()))
        image = Image.open(BytesIO(uploaded_file['content']))
    elif url_widget.value:
        if len(url_widget.value.strip().split()) > 1:
            with output:
                output.clear_output()
                print("⚠️ Please enter only one image URL.")
            return
        try:
            url = convert_drive_url(url_widget.value)
            response = requests.get(url, stream=True)
            content_type = response.headers.get("Content-Type", "")
            if "image" not in content_type:
                raise ValueError(f"URL does not point to an image. Got content type: {content_type}")
            image = Image.open(BytesIO(response.content))
        except Exception as e:
            with output:
                output.clear_output()
                print(f"❌ Failed to load image from URL: {e}")
            return

    if image:
        run_models_on_image(image)
    else:
        with output:
            output.clear_output()
            print("⚠️ No image provided.")

In [ ]:
def on_click_detector(b):
    image = None
    if upload_widget.value:
        uploaded_file = next(iter(upload_widget.value.values()))
        image = Image.open(BytesIO(uploaded_file['content']))
    elif url_widget.value:
        if len(url_widget.value.strip().split()) > 1:
            with output:
                output.clear_output()
                print("⚠️ Please enter only one image URL.")
            return
        try:
            url = convert_drive_url(url_widget.value)
            response = requests.get(url, stream=True)
            content_type = response.headers.get("Content-Type", "")
            if "image" not in content_type:
                raise ValueError(f"URL does not point to an image. Got content type: {content_type}")
            image = Image.open(BytesIO(response.content))
        except Exception as e:
            with output:
                output.clear_output()
                print(f"❌ Failed to load image from URL: {e}")
            return
    if image:
        run_detector_on_image(image)
    else:
        with output:
            output.clear_output()
            print("⚠️ No image provided.")

In [ ]:
def on_click_show_image(b):
    output.clear_output()
    with output:
        image = None
        if upload_widget.value:
            uploaded_file = next(iter(upload_widget.value.values()))
            image = Image.open(BytesIO(uploaded_file['content']))
        elif url_widget.value:
            try:
                url = convert_drive_url(url_widget.value)
                response = requests.get(url, stream=True)
                image = Image.open(BytesIO(response.content))
            except Exception as e:
                print(f"❌ Failed to load image for display: {e}")
                return
        if image:
            plt.figure(figsize=(4, 4))
            plt.imshow(image.convert("L"), cmap='gray')
            plt.title("Uploaded Image")
            plt.axis('off')
            plt.show()
        else:
            print("⚠️ No image provided for display.")

In [ ]:
title_label = widgets.HTML(
    value="""
    <div style='
        color: #4169E1;
        padding: 10px;
        border-radius: 6px;
        font-size: 20px;
        text-align: left;
        font-weight: bold;
        width: 195px;
    '>
         MEDICAL IMAGING PROJECT 💻🥽🩺
    </div>
    """
)

footer_label = widgets.HTML(
    value="""
    <div style='
        color: white;
        padding: 6px;
        font-weight: bold;
        border-radius: 6px;
        font-size: 12px;
        text-align: left;
        width: 180px;
    '>
        © Lior Cohen & Tomer Peretz ©
    </div>
    """
)

### Gradcam

> Explainability using Grad-CAM for PyTorch models

*   Compute heatmaps for the most activated regions in the model
*   Automatically choose the correct internal layer if available
*   Overlay Grad-CAM on top of the input image
*   Show results with predicted label and confidence score

In [ ]:
def generate_gradcam(model, image_tensor, target_layer=None):
    model.eval()
    gradients = []
    activations = []
    def forward_hook(module, input, output):
        activations.append(output)
    def backward_hook(module, grad_input, grad_output):
        gradients.append(grad_output[0])
    if target_layer is None:
        target_layer = list(dict(model.named_modules()).values())[-1]
    handle_fw = target_layer.register_forward_hook(forward_hook)
    handle_bw = target_layer.register_backward_hook(backward_hook)
    output = model(image_tensor)
    pred_class = output.sigmoid().squeeze().argmax().item()
    model.zero_grad()
    output[0, pred_class].backward()
    handle_fw.remove()
    handle_bw.remove()
    grad = gradients[0]
    act = activations[0]
    weights = grad.mean(dim=(2, 3), keepdim=True)
    cam = (weights * act).sum(dim=1, keepdim=True)
    cam = torch.nn.functional.relu(cam)
    cam = torch.nn.functional.interpolate(cam, size=(224, 224), mode='bilinear', align_corners=False)
    cam = cam.squeeze().detach().cpu().numpy()
    cam -= cam.min()
    cam /= cam.max()
    return cam, pred_class

In [ ]:
def show_cam_on_image(original_image: Image.Image, cam: np.ndarray, title: str = ""):
    fixed_size = (224, 224)
    image = original_image.resize(fixed_size).convert("RGB")
    image_np = np.array(image).astype(np.float32) / 255.0
    cam = cv2.resize(cam, fixed_size)
    fig, axs = plt.subplots(1, 2, figsize=(10, 4))
    axs[0].imshow(cam, cmap='jet')
    axs[0].set_title(f"{title}", fontsize=7)
    axs[0].axis('off')
    axs[1].imshow(image_np)
    axs[1].imshow(cam, cmap='jet', alpha=0.5)
    axs[1].set_title(f"{title}", fontsize=7)
    axs[1].axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
def on_click_gradcam(b):
    global output
    output.clear_output()
    with output:
        image = None
        if upload_widget.value:
            uploaded_file = next(iter(upload_widget.value.values()))
            image = Image.open(BytesIO(uploaded_file['content']))
        elif url_widget.value:
            try:
                url = convert_drive_url(url_widget.value)
                response = requests.get(url, stream=True)
                image = Image.open(BytesIO(response.content))
            except Exception as e:
                print(f"❌ Failed to load image for Grad-CAM: {e}")
                return
        if not image:
            print("⚠️ No image provided for Grad-CAM.")
            return
        tensor_img = preprocess_pil_image(image)
        for name, model_data in torch_models.items():
            torch_model = model_data["model"]
            try:
                if hasattr(torch_model, "model") and hasattr(torch_model.model, "features"):
                    target_layer = torch_model.model.features[-2]
                elif hasattr(torch_model, "features"):
                    target_layer = torch_model.features[-2]
                else:
                    print(f"⚠️ Model {name} does not have a recognizable 'features' structure.")
                    continue
                cam, cls = generate_gradcam(torch_model, tensor_img, target_layer=target_layer)
                label = torch_model.pathologies[cls] if hasattr(torch_model, 'pathologies') else f"Class {cls}"
                if label.strip() == "":
                    label = "No Finding"
                prob_percent = 0
                with torch.no_grad():
                  output1 = torch_model(tensor_img)
                  probs = torch.sigmoid(output1).numpy().flatten()
                  prob_percent = probs[cls] * 100
                show_cam_on_image(image, cam, title=f"{name} - {label} ({prob_percent:.3f}%)")
            except Exception as e:
                print(f"❌ Grad-CAM failed for {name}: {e}")

### Main

> Visual layout and interactivity

*   Create header and footer labels with custom styling
*   Build and display the user interface with all interactive buttons and inputs
*   Attach behavior handlers for each widget

In [ ]:
def interface():
  upload_widget.observe(on_upload_change, names='value')
  url_widget.observe(on_url_change, names='value')
  reset_button.on_click(on_reset_clicked)
  run_button.on_click(on_click_run)
  show_image_button.on_click(on_click_show_image)
  gradcam_button.on_click(on_click_gradcam)
  detector_button.on_click(on_click_detector)

  components = [
    footer_label,
    title_label,
    widgets.Label(""),
    url_widget,
    upload_widget,
    reset_button,
    show_image_button,
    run_button,
    detector_button,
    gradcam_button,
    output
]

  display(widgets.VBox(components))

#### Sound

> Optional audio for UI feedback
*   Download and play a short mp3 file from Google Drive
*  SpongeBob SquarePants says "I'm ready!"

In [ ]:
def play():
  file_id = '1Fao_XGFln-nqCkLO-3zrozDswgltjbEs'
  downloaded_file = drive.CreateFile({'id': file_id})
  downloaded_file.GetContentFile('sound.mp3')
  return Audio('sound.mp3', autoplay=True)

In [ ]:
play()

# Interface

> You can choose how to load an image:

*   Select a file from your local computer

*   Or enter a Google Drive URL, like: (https://drive.google.com/file/d/1Kq9puOMVF1HSD8TkdVilQPhmiuAolmAZ/view)


> Options:


1.   Attach an image via URL
2.   Upload an image
3.   Reset the image
4.   View the image
5.   Run CNN models
6.   Run Disease Detector
6.   Run Explainability methods



In [ ]:
interface()